In [0]:
%run ../Notebooks/00_Configuration

Configuration Loaded Successfully


In [0]:
%run ../framework/01_Utility_Functions

Configuration Loaded Successfully


In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("pipeline_run_id", "")

import uuid
from datetime import datetime
from pyspark.sql.functions import (
    col, lit, max as spark_max, upper,
    to_timestamp, try_to_timestamp, coalesce
)
from pyspark.sql.types import TimestampType

pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

print("=" * 80)
print("ENTERPRISE GENERIC BRONZE LOADER V4")
print("=" * 80)
print("Pipeline Run :", pipeline_run_id)

ENTERPRISE GENERIC BRONZE LOADER V4
Pipeline Run : 


In [0]:
NOTEBOOK_NAME = "Enterprise_Generic_Bronze_Loader_v4"

def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def print_info(name, value):
    print(f"{name:<30}: {value}")

def print_success(message):
    print(f"SUCCESS : {message}")

def print_warning(message):
    print(f"WARNING : {message}")

def print_error(message):
    print(f"ERROR   : {message}")

def get_duplicate_count(df, primary_key):
    return (
        df.groupBy(primary_key)
          .count()
          .filter("count > 1")
          .count()
    )

def get_null_count(df, primary_key):
    return df.filter(col(primary_key).isNull()).count()

# ----------------------------------------------------------------------------
# FIX: safe_to_timestamp()
#
# Original code called to_timestamp(col(watermark_column)) with no format.
# Under ANSI mode, an unparseable string (e.g. Claim's "11-04-2022 16:40",
# which is dd-MM-yyyy, not the default-inferred format) throws
# CAST_INVALID_INPUT and fails the whole table instead of just that row.
#
# This tries the default parse first (keeps Agent/Branch/Customer/Policy
# working exactly as before, since their watermark columns already parse
# fine), then falls back to dd-MM-yyyy HH:mm (Claim's format) using
# try_to_timestamp, which returns NULL on failure instead of raising.
# Add more fallback formats here if another table needs one later.
# ----------------------------------------------------------------------------
def safe_to_timestamp(column):
    return coalesce(
        try_to_timestamp(column),
        try_to_timestamp(column, lit("dd-MM-yyyy HH:mm"))
    )

In [0]:
print_header("READING ALL ACTIVE METADATA")

metadata_df = (
    spark.table(f"{catalog_name}.{metadata_schema}.bronze_config")
         .filter(upper(col("active")) == "Y")
         .orderBy("table_name")
)

metadata_rows = metadata_df.collect()

if len(metadata_rows) == 0:
    raise Exception("No active tables found in bronze_config.")

print_success(f"{len(metadata_rows)} active table(s) found.")
for r in metadata_rows:
    print_info("Active Table", r["table_name"])


READING ALL ACTIVE METADATA
SUCCESS : 5 active table(s) found.
Active Table                  : Agent
Active Table                  : Branch
Active Table                  : Claim
Active Table                  : Customer
Active Table                  : Policy


In [0]:
print_header("PROCESSING ALL ACTIVE TABLES")
results_summary = []
for config in metadata_rows:
    table_name = config["table_name"]
    load_id = str(uuid.uuid4())
    print_header(f"PROCESSING {table_name}")
    try:
        # ---------------- Metadata ----------------
        source_folder = config["source_folder"]
        target_table = config["target_table"]
        gold_table = config["gold_table"]
        history_table = config["history_table"]
        file_format = (config["file_format"] or "").lower().strip()
        load_strategy = (config["load_strategy"] or "").upper().strip()
        primary_key = config["primary_key"]
        source_system = config["source_system"]
        watermark_column = config["watermark_column"]
        compare_columns = config["compare_columns"]
        scd_enabled = config["scd_enabled"]

        print_info("Source Folder", source_folder)
        print_info("Target Table", target_table)
        print_info("File Format", file_format)
        print_info("Primary Key", primary_key)
        print_info("Load Strategy", load_strategy)
        print_info("Watermark Column", watermark_column)

        # ---------------- Validate metadata ----------------
        mandatory_fields = {
            "source_folder": source_folder,
            "target_table": target_table,
            "file_format": file_format,
            "load_strategy": load_strategy,
            "primary_key": primary_key,
            "source_system": source_system
        }

        for field_name, field_value in mandatory_fields.items():
            if field_value is None or str(field_value).strip() == "":
                raise Exception(f"Mandatory metadata field '{field_name}' is missing.")

        if file_format not in ["csv", "json"]:
            raise Exception(f"Unsupported file format '{file_format}'.")

        if load_strategy not in ["FULL", "INCREMENTAL"]:
            raise Exception(f"Unsupported load strategy '{load_strategy}'.")

        print_success("Metadata validation completed.")

        # ---------------- Watermark ----------------
        watermark_table = f"{catalog_name}.{metadata_schema}.pipeline_watermark"
        watermark_record = (
            spark.table(watermark_table)
            .filter(col("table_name") == table_name)
            .first()
        )

        if watermark_record is None:
            print_warning("No watermark found. Initializing first load.")
            last_watermark = datetime(1900, 1, 1)
            spark.createDataFrame(
                [(table_name, last_watermark)], ["table_name", "last_watermark"]
            ).write.mode("append").saveAsTable(watermark_table)
        else:
            last_watermark = watermark_record["last_watermark"]

        print_info("Current Watermark", last_watermark)

        # ---------------- Source path ----------------
        source_path = f"{landing_path}{source_folder}"
        print_info("Landing Path", source_path)

        try:
            source_files = dbutils.fs.ls(source_path)
        except Exception:
            raise Exception(f"Landing path does not exist : {source_path}")

        if len(source_files) == 0:
            raise Exception(f"No files found in {source_path}")

        print_success(f"{len(source_files)} file(s) discovered.")

        # ---------------- Read ----------------
        if file_format == "csv":
            df = (
                spark.read
                .format("csv")
                .option("header", True)
                .option("inferSchema", True)
                .load(source_path)
            )
            target_table_fqn = f"{catalog_name}.{bronze_schema}.{target_table}"
            try:
                existing_schema = spark.table(target_table_fqn).schema
                existing_types = {f.name: f.dataType for f in existing_schema}
                for field in df.schema:
                    if field.name in existing_types and field.dataType != existing_types[field.name]:
                        print_warning(
                            f"[{table_name}] Column '{field.name}' inferred as "
                            f"{field.dataType.simpleString()} this run, casting to match "
                            f"existing table type {existing_types[field.name].simpleString()}"
                        )
                        # FIX: a plain .cast() to TimestampType throws CAST_INVALID_INPUT
                        # under ANSI mode for malformed strings (e.g. Claim's dd-MM-yyyy
                        # HH:mm values), exactly like the original watermark bug. Route
                        # timestamp targets through safe_to_timestamp() so this loop can
                        # never reintroduce that failure; all other types keep a plain cast.
                        if isinstance(existing_types[field.name], TimestampType):
                            df = df.withColumn(field.name, safe_to_timestamp(col(field.name)))
                        else:
                            df = df.withColumn(field.name, col(field.name).cast(existing_types[field.name]))
            except AnalysisException:
                # target table doesn't exist yet (first run) — nothing to reconcile against
                pass
        else:
            df = (
                spark.read
                .format("json")
                .load(source_path)
            )

        rows_before_filter = df.count()
        print_info("Rows Read", rows_before_filter)

        # ---------------- Incremental filter ----------------
        if load_strategy == "INCREMENTAL" and watermark_column and watermark_column.strip() != "":
            print_info("Load Type", "INCREMENTAL")
            # FIX: was to_timestamp(col(watermark_column)) with no format,
            # which threw CAST_INVALID_INPUT under ANSI mode for formats
            # like Claim's "dd-MM-yyyy HH:mm". safe_to_timestamp() tries the
            # default parse, falls back to dd-MM-yyyy HH:mm, and never throws.
            df = df.withColumn("_watermark", safe_to_timestamp(col(watermark_column)))
            unparsed_count = df.filter(col("_watermark").isNull()).count()
            if unparsed_count > 0:
                print_warning(
                    f"{unparsed_count} row(s) had an unparseable '{watermark_column}' "
                    f"value and will be excluded from this incremental load."
                )
            df = df.filter(col("_watermark") > lit(last_watermark))
            df = df.drop("_watermark")
        else:
            print_info("Load Type", "FULL")

        rows_after_filter = df.count()
        print_info("Rows After Filter", rows_after_filter)

        # ---------------- Data quality ----------------
        duplicate_count = get_duplicate_count(df, primary_key)
        null_key_count = get_null_count(df, primary_key)

        print_info("Duplicate Keys", duplicate_count)
        print_info("Null Keys", null_key_count)

        if rows_after_filter == 0:
            print_warning("No new records found.")
        else:
            print_success(f"{rows_after_filter} rows ready for Bronze.")

        # ---------------- Add metadata columns (ONCE, not twice) ----------------
        df = add_metadata(
            df=df,
            source_system=source_system,
            pipeline_run_id=pipeline_run_id,
            load_id=load_id
        )

        if "_metadata" in df.columns:
            df = df.drop("_metadata")

        # ---------------- Write ----------------
        bronze_table = f"{catalog_name}.{bronze_schema}.{target_table}"
        print_info("Bronze Table", bronze_table)

        write_mode = "overwrite" if load_strategy == "FULL" else "append"
        writer = df.write.format("delta").mode(write_mode)

        if write_mode == "overwrite":
            writer = writer.option("overwriteSchema", "true")

        writer.saveAsTable(bronze_table)

        rows_written = rows_after_filter
        bronze_total_rows = spark.table(bronze_table).count()

        print_success("Bronze Load Completed")
        print_info("Rows Written", rows_written)
        print_info("Bronze Total Rows", bronze_total_rows)

        # ---------------- Watermark update ----------------
        if load_strategy == "INCREMENTAL" and rows_written > 0:
            # FIX: same safe_to_timestamp() used here, so advancing the
            # watermark can't throw on the same malformed-format issue.
            new_watermark = (
                df.withColumn("_watermark", safe_to_timestamp(col(watermark_column)))
                .agg(spark_max("_watermark"))
                .collect()[0][0]
            )

            spark.sql(f"""
                UPDATE {catalog_name}.{metadata_schema}.pipeline_watermark
                SET last_watermark = TIMESTAMP('{new_watermark}')
                WHERE table_name = '{table_name}'
            """)

            print_success("Watermark Updated")
            print_info("New Watermark", new_watermark)
        else:
            print_warning("Watermark Not Updated")

        # ---------------- Audit (success) ----------------
        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=target_table,
            load_type=load_strategy,
            rows_read=rows_before_filter,
            rows_written=rows_written,
            status="SUCCESS",
            error_message="",
            pipeline_run_id=pipeline_run_id
        )
        print_success("Audit Written")

        results_summary.append({
            "table_name": table_name,
            "status": "SUCCESS",
            "rows_read": rows_before_filter,
            "rows_written": rows_written,
            "error": ""
        })

    except Exception as ex:
        print_error(str(ex))
        write_audit(
            pipeline_name=NOTEBOOK_NAME,
            table_name=table_name,
            load_type=config["load_strategy"] if config["load_strategy"] else "UNKNOWN",
            rows_read=0,
            rows_written=0,
            status="FAILED",
            error_message=str(ex),
            pipeline_run_id=pipeline_run_id
        )
        results_summary.append({
            "table_name": table_name,
            "status": "FAILED",
            "rows_read": 0,
            "rows_written": 0,
            "error": str(ex)
        })
        # One table's failure does not stop the other tables from loading
        continue


PROCESSING ALL ACTIVE TABLES

PROCESSING Agent
Source Folder                 : agent
Target Table                  : agent
File Format                   : csv
Primary Key                   : agent_id
Load Strategy                 : FULL
Watermark Column              : create_timestamp
SUCCESS : Metadata validation completed.
Current Watermark             : 1900-01-01 00:00:00
Landing Path                  : abfss://landing@adlsinsurancedevstorage.dfs.core.windows.net/agent
SUCCESS : 2 file(s) discovered.
Rows Read                     : 2005
Load Type                     : FULL
Rows After Filter             : 2005
Duplicate Keys                : 1000
Null Keys                     : 0
SUCCESS : 2005 rows ready for Bronze.
Bronze Table                  : dbw_insurance.insurance_bronze.agent
SUCCESS : Bronze Load Completed
Rows Written                  : 2005
Bronze Total Rows             : 2005
SUCCESS : Audit Written

PROCESSING Branch
Source Folder                 : branch
Target Table

In [0]:
# print_header("PROCESSING ALL ACTIVE TABLES")

# results_summary = []

# for config in metadata_rows:

#     table_name = config["table_name"]
#     load_id = str(uuid.uuid4())

#     print_header(f"PROCESSING {table_name}")

#     try:
#         # ---------------- Metadata ----------------
#         source_folder    = config["source_folder"]
#         target_table     = config["target_table"]
#         gold_table       = config["gold_table"]
#         history_table    = config["history_table"]
#         file_format      = (config["file_format"] or "").lower().strip()
#         load_strategy    = (config["load_strategy"] or "").upper().strip()
#         primary_key      = config["primary_key"]
#         source_system    = config["source_system"]
#         watermark_column = config["watermark_column"]
#         compare_columns  = config["compare_columns"]
#         scd_enabled      = config["scd_enabled"]

#         print_info("Source Folder", source_folder)
#         print_info("Target Table", target_table)
#         print_info("File Format", file_format)
#         print_info("Primary Key", primary_key)
#         print_info("Load Strategy", load_strategy)
#         print_info("Watermark Column", watermark_column)

#         # ---------------- Validate metadata ----------------
#         mandatory_fields = {
#             "source_folder": source_folder,
#             "target_table": target_table,
#             "file_format": file_format,
#             "load_strategy": load_strategy,
#             "primary_key": primary_key,
#             "source_system": source_system
#         }
#         for field_name, field_value in mandatory_fields.items():
#             if field_value is None or str(field_value).strip() == "":
#                 raise Exception(f"Mandatory metadata field '{field_name}' is missing.")

#         if file_format not in ["csv", "json"]:
#             raise Exception(f"Unsupported file format '{file_format}'.")
#         if load_strategy not in ["FULL", "INCREMENTAL"]:
#             raise Exception(f"Unsupported load strategy '{load_strategy}'.")

#         print_success("Metadata validation completed.")

#         # ---------------- Watermark ----------------
#         watermark_table = f"{catalog_name}.{metadata_schema}.pipeline_watermark"

#         watermark_record = (
#             spark.table(watermark_table)
#                  .filter(col("table_name") == table_name)
#                  .first()
#         )

#         if watermark_record is None:
#             print_warning("No watermark found. Initializing first load.")
#             last_watermark = datetime(1900, 1, 1)
#             spark.createDataFrame(
#                 [(table_name, last_watermark)],
#                 ["table_name", "last_watermark"]
#             ).write.mode("append").saveAsTable(watermark_table)
#         else:
#             last_watermark = watermark_record["last_watermark"]

#         print_info("Current Watermark", last_watermark)

#         # ---------------- Source path ----------------
#         source_path = f"{landing_path}{source_folder}"
#         print_info("Landing Path", source_path)

#         try:
#             source_files = dbutils.fs.ls(source_path)
#         except Exception:
#             raise Exception(f"Landing path does not exist : {source_path}")

#         if len(source_files) == 0:
#             raise Exception(f"No files found in {source_path}")

#         print_success(f"{len(source_files)} file(s) discovered.")

#         # ---------------- Read ----------------
#         if file_format == "csv":
#             df = (spark.read   
#                   .format("csv")
#                   .format("csv")
#                   .option("header", True)
#                   .option("inferSchema", True)
#                   .load(source_path)
#                     target_table_fqn = f"{catalog_name}.{bronze_schema}.{target_table}"
# try:
#     existing_schema = spark.table(target_table_fqn).schema
#     existing_types = {f.name: f.dataType for f in existing_schema}
#     for field in df.schema:
#         if field.name in existing_types and field.dataType != existing_types[field.name]:
#             print_warning(
#                 f"[{table_name}] Column '{field.name}' inferred as "
#                 f"{field.dataType.simpleString()} this run, casting to match "
#                 f"existing table type {existing_types[field.name].simpleString()}"
#             )
#             df = df.withColumn(field.name, col(field.name).cast(existing_types[field.name]))
# except AnalysisException:
#     # target table doesn't exist yet (first run) — nothing to reconcile against
#     pass
#             )
#         else:
#              df = (spark.read
#                 .format("json")
#                 .load(source_path)
#             )

#         rows_before_filter = df.count()
#         print_info("Rows Read", rows_before_filter)

#         # ---------------- Incremental filter ----------------
#         if load_strategy == "INCREMENTAL" and watermark_column and watermark_column.strip() != "":
#             print_info("Load Type", "INCREMENTAL")

#             # FIX: was to_timestamp(col(watermark_column)) with no format,
#             # which threw CAST_INVALID_INPUT under ANSI mode for formats
#             # like Claim's "dd-MM-yyyy HH:mm". safe_to_timestamp() tries the
#             # default parse, falls back to dd-MM-yyyy HH:mm, and never throws.
#             df = df.withColumn("_watermark", safe_to_timestamp(col(watermark_column)))

#             unparsed_count = df.filter(col("_watermark").isNull()).count()
#             if unparsed_count > 0:
#                 print_warning(
#                     f"{unparsed_count} row(s) had an unparseable '{watermark_column}' "
#                     f"value and will be excluded from this incremental load."
#                 )

#             df = df.filter(col("_watermark") > lit(last_watermark))
#             df = df.drop("_watermark")
#         else:
#             print_info("Load Type", "FULL")

#         rows_after_filter = df.count()
#         print_info("Rows After Filter", rows_after_filter)

#         # ---------------- Data quality ----------------
#         duplicate_count = get_duplicate_count(df, primary_key)
#         null_key_count = get_null_count(df, primary_key)
#         print_info("Duplicate Keys", duplicate_count)
#         print_info("Null Keys", null_key_count)

#         if rows_after_filter == 0:
#             print_warning("No new records found.")
#         else:
#             print_success(f"{rows_after_filter} rows ready for Bronze.")

#         # ---------------- Add metadata columns (ONCE, not twice) ----------------
#         df = add_metadata(
#             df=df,
#             source_system=source_system,
#             pipeline_run_id=pipeline_run_id,
#             load_id=load_id
#         )
#         if "_metadata" in df.columns:
#             df = df.drop("_metadata")

#         # ---------------- Write ----------------
#         bronze_table = f"{catalog_name}.{bronze_schema}.{target_table}"
#         print_info("Bronze Table", bronze_table)

#         write_mode = "overwrite" if load_strategy == "FULL" else "append"

#         writer = df.write.format("delta").mode(write_mode)
#         if write_mode == "overwrite":
#             writer = writer.option("overwriteSchema", "true")

#         writer.saveAsTable(bronze_table)

#         rows_written = rows_after_filter
#         bronze_total_rows = spark.table(bronze_table).count()

#         print_success("Bronze Load Completed")
#         print_info("Rows Written", rows_written)
#         print_info("Bronze Total Rows", bronze_total_rows)

#         # ---------------- Watermark update ----------------
#         if load_strategy == "INCREMENTAL" and rows_written > 0:
#             # FIX: same safe_to_timestamp() used here, so advancing the
#             # watermark can't throw on the same malformed-format issue.
#             new_watermark = (
#                 df.withColumn("_watermark", safe_to_timestamp(col(watermark_column)))
#                   .agg(spark_max("_watermark"))
#                   .collect()[0][0]
#             )

#             spark.sql(f"""
#                 UPDATE {catalog_name}.{metadata_schema}.pipeline_watermark
#                 SET last_watermark = TIMESTAMP('{new_watermark}')
#                 WHERE table_name = '{table_name}'
#             """)

#             print_success("Watermark Updated")
#             print_info("New Watermark", new_watermark)
#         else:
#             print_warning("Watermark Not Updated")

#         # ---------------- Audit (success) ----------------
#         write_audit(
#             pipeline_name=NOTEBOOK_NAME,
#             table_name=target_table,
#             load_type=load_strategy,
#             rows_read=rows_before_filter,
#             rows_written=rows_written,
#             status="SUCCESS",
#             error_message="",
#             pipeline_run_id=pipeline_run_id
#         )
#         print_success("Audit Written")

#         results_summary.append({
#             "table_name": table_name,
#             "status": "SUCCESS",
#             "rows_read": rows_before_filter,
#             "rows_written": rows_written,
#             "error": ""
#         })

#     except Exception as ex:

#         print_error(str(ex))

#         write_audit(
#             pipeline_name=NOTEBOOK_NAME,
#             table_name=table_name,
#             load_type=config["load_strategy"] if config["load_strategy"] else "UNKNOWN",
#             rows_read=0,
#             rows_written=0,
#             status="FAILED",
#             error_message=str(ex),
#             pipeline_run_id=pipeline_run_id
#         )

#         results_summary.append({
#             "table_name": table_name,
#             "status": "FAILED",
#             "rows_read": 0,
#             "rows_written": 0,
#             "error": str(ex)
#         })

#         # One table's failure does not stop the other tables from loading
#         continue

In [0]:
print("\n")
print("=" * 90)
print("ENTERPRISE GENERIC BRONZE LOADER V4 COMPLETED")
print("=" * 90)

for r in results_summary:
    line = f"{r['table_name']:<15} : {r['status']:<8} Read={r['rows_read']:<6} Written={r['rows_written']:<6}"
    if r["status"] == "FAILED":
        line += f"  ERROR: {r['error']}"
    print(line)

success_count = len([r for r in results_summary if r["status"] == "SUCCESS"])
failed_count = len([r for r in results_summary if r["status"] == "FAILED"])

print("=" * 90)
print_success(f"Processed {len(results_summary)} tables. Success={success_count}  Failed={failed_count}")
print("=" * 90)



ENTERPRISE GENERIC BRONZE LOADER V4 COMPLETED
Agent           : SUCCESS  Read=2005   Written=2005  
Branch          : SUCCESS  Read=1000   Written=1000  
Claim           : SUCCESS  Read=1000   Written=0     
Customer        : SUCCESS  Read=2010   Written=0     
Policy          : SUCCESS  Read=1000   Written=1000  
SUCCESS : Processed 5 tables. Success=5  Failed=0


In [0]:
%sql
DESCRIBE TABLE dbw_insurance.insurance_bronze.claim

col_name,data_type,comment
claim_id,int,null
policy_id,int,null
date_of_claim,timestamp,null
claim_amount,double,null
claim_status,string,null
LastUpdatedTimeStamp,timestamp,null
ingestion_timestamp,timestamp,null
source_file_name,string,null
pipeline_run_id,string,null
load_id,string,null
